# Fase 1 — Modelo predictivo: detección de intrusiones en red (UNSW-NB15)

**Curso:** Modelos y Simulación de Sistemas I — UdeA, 2026-2
**Equipo:** Adrian, Felipe Cetina, Bryan Medrano

## 1. Descripción del problema y del dataset

- **Problema:** clasificar cada flujo de tráfico de red como **normal (0)** o **ataque (1)**. Clasificación binaria.
- **Motivación:** una red recibe miles de conexiones por minuto y solo una fracción son intentos de ataque (escaneos, DoS, exploits, puertas traseras, entre otros). Revisarlas manualmente no es viable a esa escala; un modelo que las distinga automáticamente permite priorizar la atención del equipo de seguridad y reaccionar antes de que un ataque tenga éxito.
- **Dataset:** UNSW-NB15 (Australian Centre for Cyber Security), versión curada `UNSW_NB15_training-set.csv`. Cada fila es un flujo de red descrito por variables de tráfico (duración, bytes, paquetes, TTL, protocolo, servicio, estado…).
- **Variable objetivo:** `label`.
- **Columnas excluidas:** `id` (identificador) y `attack_cat` (tipo de ataque: se deriva directamente del objetivo, usarla sería fuga de información).
- **Métrica principal:** F1 de la clase ataque, porque importa tanto no dejar pasar ataques (recall) como no generar falsas alarmas (precision).
- **Hallazgos del EDA y decisiones tomadas:**
  - El archivo trae 82.332 filas y 45 columnas, sin valores nulos reales en ninguna columna. El profesor aprobó el dataset aunque no cumple estrictamente el requisito de 0,1 %-2 % de nulos, porque no es una restricción crítica del curso.
  - `service = "-"` aparece en el 65 % de las filas y representa "servicio no identificado"; se trata como categoría legítima, no como dato faltante.
  - Se detectaron **28.380 filas duplicadas (34,5 %)** al comparar solo las columnas predictoras y el objetivo (`id` es un simple contador de fila). Se eliminaron antes de dividir en train/test (sección 4.1.1).
  - El objetivo está moderadamente desbalanceado tras la limpieza: 63,4 % normal / 36,6 % ataque.
  - Las variables más asociadas al objetivo son `sttl` y `dttl` (TTL de origen y destino), `dload` (carga del destino) y los contadores de conexiones recientes `ct_dst_sport_ltm` y `ct_state_ttl`.
  - `proto` tiene 131 categorías, la mayoría muy poco frecuentes (`tcp` y `udp` concentran el 85 % de las filas); se codifican todas con `OneHotEncoder(handle_unknown="ignore")` para no perder información y para tolerar protocolos nuevos en producción.

> Este notebook se ejecuta de principio a fin con **Reiniciar y ejecutar todo**. Todas las transformaciones se ajustan **solo con train** dentro de un `Pipeline`, así que no hay fuga de información.

## 2. Configuración

Toda la configuración y las **columnas de entrada** están definidas aquí, en un único lugar. Estas listas se reutilizarán en `predict.py` (fase 2) y en la validación de la API (fase 3).

In [1]:
import os
import sys
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

# --- Datos ---
DATA_PATH = Path(os.environ.get("DATA_PATH", "../data/raw/UNSW_NB15_training-set.csv"))
TARGET = "label"
COLUMNAS_EXCLUIR = ["id", "attack_cat"]  # id = identificador; attack_cat = fuga del objetivo

# --- Columnas de entrada del modelo (contrato para fase-2 y fase-3) ---
COLS_CATEGORICAS = ["proto", "service", "state"]
COLS_NUMERICAS = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sttl", "dttl", "sload", "dload",
    "sloss", "dloss", "sinpkt", "dinpkt", "sjit", "djit", "swin", "stcpb", "dtcpb", "dwin",
    "tcprtt", "synack", "ackdat", "smean", "dmean", "trans_depth", "response_body_len",
    "ct_srv_src", "ct_state_ttl", "ct_dst_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "is_ftp_login", "ct_ftp_cmd", "ct_flw_http_mthd", "ct_src_ltm",
    "ct_srv_dst", "is_sm_ips_ports",
]
COLS_ENTRADA = COLS_NUMERICAS + COLS_CATEGORICAS

# --- Experimento ---
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 3
VERSION_MODELO = "v1"
MODELS_DIR = Path("models")
MAX_MB_MODELO = 50  # por encima de esto no conviene subirlo a GitHub

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
print("Python", sys.version.split()[0], "| sklearn", sklearn.__version__, "| pandas", pd.__version__)


Python 3.13.7 | sklearn 1.9.1 | pandas 3.0.6


## 3. Carga de datos

El CSV se coloca manualmente en `data/raw/` (ver `data/README.md`). Si el notebook corre en Google Colab y el archivo no está, se pide subirlo.

In [2]:
if not DATA_PATH.exists() and "google.colab" in sys.modules:
    from google.colab import files
    print(f"No se encontró {DATA_PATH}. Sube UNSW_NB15_training-set.csv:")
    subido = files.upload()
    DATA_PATH = Path(next(iter(subido)))

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Coloca el CSV en {DATA_PATH.resolve()} (ver data/README.md)")

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Archivo: {DATA_PATH} | {df.shape[0]:,} filas x {df.shape[1]} columnas")

# El contrato de columnas debe coincidir con el CSV
faltantes = set(COLS_ENTRADA + [TARGET]) - set(df.columns)
no_usadas = set(df.columns) - set(COLS_ENTRADA + [TARGET] + COLUMNAS_EXCLUIR)
assert not faltantes, f"Columnas esperadas que no están en el CSV: {faltantes}"
print("Columnas del CSV no usadas ni excluidas:", no_usadas or "ninguna")
df.head()


Archivo: ../data/raw/UNSW_NB15_training-set.csv | 82,332 filas x 45 columnas
Columnas del CSV no usadas ni excluidas: ninguna


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


## 4. Análisis exploratorio (EDA)

### 4.1 Dimensiones y tipos

In [3]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes.astype(str).value_counts())
print("\nFilas duplicadas:", df.duplicated(subset=COLS_ENTRADA + [TARGET]).sum())


Dimensiones: (82332, 45)

Tipos de datos:
int64      30
float64    11
str         4
Name: count, dtype: int64



Filas duplicadas: 28380


### 4.1.1 Eliminación de duplicados

Se detectaron duplicados considerando solo las columnas predictoras y el objetivo (`id` es un simple contador de fila, así que se ignora para esta comparación). Se eliminan antes de continuar con el resto del EDA, para que las estadísticas y gráficas siguientes reflejen el dataset ya limpio.

In [4]:
filas_antes = len(df)
df = df.drop_duplicates(subset=COLS_ENTRADA + [TARGET]).reset_index(drop=True)
filas_despues = len(df)

print(f"Filas antes de eliminar duplicados: {filas_antes:,}")
print(f"Duplicados eliminados:              {filas_antes - filas_despues:,} ({(filas_antes - filas_despues) / filas_antes:.1%})")
print(f"Filas después:                      {filas_despues:,}")


Filas antes de eliminar duplicados: 82,332
Duplicados eliminados:              28,380 (34.5%)
Filas después:                      53,952


### 4.2 Valores nulos

Se separan los NaN reales de los marcadores de texto como `"-"`. En UNSW-NB15, `service = "-"` significa *servicio no identificado*, así que se trata como una categoría legítima y no como un nulo.

In [5]:
nulos = pd.DataFrame({
    "NaN_%": df[COLS_ENTRADA].isna().mean() * 100,
    "marcador_-_%": pd.Series({c: (df[c].astype(str).str.strip() == "-").mean() * 100
                              for c in COLS_CATEGORICAS}),
}).fillna(0)
nulos = nulos[(nulos > 0).any(axis=1)].round(3)
print(nulos if not nulos.empty else "Sin nulos ni marcadores en las columnas de entrada")

en_rango = nulos.index[nulos["NaN_%"].between(0.1, 2)] if not nulos.empty else []
print("\nPredictoras con 0,1 %-2 % de NaN (requisito del curso):", list(en_rango) or "NINGUNA")


         NaN_%  marcador_-_%
service    0.0        64.999

Predictoras con 0,1 %-2 % de NaN (requisito del curso): NINGUNA


### 4.3 Distribución del objetivo

In [6]:
dist = df[TARGET].value_counts()
print(pd.DataFrame({"n": dist, "%": (dist / len(df) * 100).round(2)}))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
dist.plot.bar(ax=ax[0], title="Distribución de label (0 = normal, 1 = ataque)", rot=0)
if "attack_cat" in df.columns:  # solo para entender el problema; NO se usa como predictora
    df["attack_cat"].value_counts().plot.barh(ax=ax[1], title="Tipos de ataque (attack_cat)")
plt.tight_layout()
plt.show()


           n     %
label             
0      34206  63.4
1      19746  36.6


/tmp/ipykernel_106972/526566357.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.4 Estadísticas descriptivas

In [7]:
df[COLS_NUMERICAS].describe().T

,count,mean,std,min,25%,50%,75%,max
dur,53952.0,1.022065e+00,3.918370e+00,0.0,0.004234,3.888760e-01,9.371102e-01,5.999999e+01
spkts,53952.0,2.586710e+01,1.611453e+02,1.0,4.000000,1.000000e+01,1.600000e+01,1.064600e+04
dpkts,53952.0,2.596417e+01,1.416998e+02,0.0,2.000000,8.000000e+00,1.800000e+01,1.101800e+04
sbytes,53952.0,1.130989e+04,2.072115e+05,24.0,532.000000,9.000000e+02,2.158000e+03,1.435577e+07
dbytes,53952.0,1.985651e+04,1.865861e+05,0.0,178.000000,3.540000e+02,1.824000e+03,1.465753e+07
rate,53952.0,3.622747e+04,1.121800e+05,0.0,24.102463,6.549285e+01,3.295167e+03,1.000000e+06
sttl,53952.0,1.509151e+02,1.067506e+02,0.0,31.000000,2.540000e+02,2.540000e+02,2.550000e+02
dttl,53952.0,1.363606e+02,1.172719e+02,0.0,29.000000,2.520000e+02,2.520000e+02,2.530000e+02
sload,53952.0,4.420963e+07,1.947544e+08,0.0,8325.015869,4.008112e+04,9.427746e+05,5.268000e+09
dload,53952.0,9.581139e+05,2.900194e+06,0.0,1785.648072,5.699247e+03,2.483357e+05,2.082111e+07


In [8]:
for col in COLS_CATEGORICAS:
    print(f"{col}: {df[col].nunique()} categorías | top 5: {df[col].value_counts().head(5).to_dict()}")


proto: 131 categorías | top 5: {'tcp': 40527, 'udp': 12523, 'arp': 298, 'unas': 199, 'ospf': 74}
service: 13 categorías | top 5: {'-': 35068, 'http': 7900, 'dns': 6091, 'smtp': 1759, 'ftp': 1291}
state: 7 categorías | top 5: {'FIN': 37323, 'INT': 8718, 'CON': 6697, 'REQ': 1208, 'ACC': 4}


### 4.5 Gráficas

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, COLS_CATEGORICAS):
    top = df[col].value_counts().head(10).index
    (df[df[col].isin(top)].groupby(col)[TARGET].mean().sort_values()
       .plot.barh(ax=ax, title=f"Proporción de ataques por {col} (top 10)"))
plt.tight_layout()
plt.show()

# Variables numéricas muy sesgadas: se muestran en escala log1p
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["sbytes", "dur", "sttl"]):
    sns.histplot(data=df, x=np.log1p(df[col].clip(lower=0)), hue=TARGET, bins=50,
                 stat="density", common_norm=False, ax=ax)
    ax.set_xlabel(f"log1p({col})")
plt.tight_layout()
plt.show()


/tmp/ipykernel_106972/2073499241.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_106972/2073499241.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
corr = df[COLS_NUMERICAS + [TARGET]].corr(numeric_only=True)
top_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(15)
print("Top 15 |correlación| con el objetivo:")
print(top_corr.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(df[top_corr.index.tolist() + [TARGET]].corr(), cmap="coolwarm", center=0)
plt.title("Correlaciones entre las 15 variables más asociadas al objetivo")
plt.show()


Top 15 |correlación| con el objetivo:
sttl                0.363
dttl                0.277
dload               0.238
ct_dst_sport_ltm    0.232
ct_state_ttl        0.188
ct_src_dport_ltm    0.166
rate                0.131
swin                0.077
dmean               0.070
ackdat              0.065
smean               0.064
ct_dst_src_ltm      0.057
sinpkt              0.055
is_sm_ips_ports     0.054
trans_depth         0.053
Name: label, dtype: float64


/tmp/ipykernel_106972/2866332073.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Separación train/test

**Antes de cualquier transformación.** Estratificada por el objetivo y con `random_state` fijo. Desde aquí, el conjunto de test solo se usa para la evaluación final.

In [11]:
X = df[COLS_ENTRADA]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("Proporción de ataques -> train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))


Train: (43161, 42) | Test: (10791, 42)
Proporción de ataques -> train: 0.366 | test: 0.366


## 6. Pipeline de preprocesamiento + modelo

Todo el preprocesamiento vive dentro de un único `Pipeline`, de modo que imputadores, escalador y codificador se ajustan solo con los datos que recibe `fit` (train o cada fold de validación cruzada).

- **Numéricas:** imputación por mediana + `StandardScaler`.
- **Categóricas:** imputación por moda + `OneHotEncoder(handle_unknown="ignore")`, para que categorías nuevas en producción no rompan el modelo.

In [12]:
def construir_pipeline(estimador):
    preprocesador = ColumnTransformer([
        ("num", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), COLS_NUMERICAS),
        ("cat", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), COLS_CATEGORICAS),
    ])
    return Pipeline([("prep", preprocesador), ("modelo", estimador)])


modelos = {
    "logreg": LogisticRegression(max_iter=2000),
    # Profundidad acotada para que el modelo serializado sea compacto
    "random_forest": RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2,
                                            n_jobs=-1, random_state=RANDOM_STATE),
}
construir_pipeline(modelos["logreg"])


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output

### 6.1 Comparación de modelos con validación cruzada (solo train)

El modelo se elige con validación cruzada sobre train. Así el test no participa en la selección y su métrica es una estimación honesta.

In [13]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
resultados_cv = {}
for nombre, estimador in modelos.items():
    scores = cross_val_score(construir_pipeline(estimador), X_train, y_train,
                             cv=cv, scoring="f1", n_jobs=1)
    resultados_cv[nombre] = scores
    print(f"{nombre:>14}: F1 CV = {scores.mean():.4f} ± {scores.std():.4f}")

mejor = max(resultados_cv, key=lambda k: resultados_cv[k].mean())
print("\nModelo seleccionado:", mejor)


        logreg: F1 CV = 0.8475 ± 0.0056


 random_forest: F1 CV = 0.9360 ± 0.0038

Modelo seleccionado: random_forest


### 6.2 Entrenamiento final

El pipeline seleccionado se reentrena con todo el conjunto de train.

In [14]:
pipe_final = construir_pipeline(modelos[mejor])
pipe_final.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](42,)","['dur','spkts','dpkts',...,'proto','service','state']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,42
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwer

### 6.3 Evidencia de que el preprocesamiento ocurrió

El `Pipeline` se ajustó **solo con `X_train`** en la celda anterior. Para dejar evidencia de la limpieza y transformación, se inspecciona aquí el resultado de esa transformación ya ajustada — **no se vuelve a hacer `fit`**, así que no hay fuga de información.

In [15]:
from scipy.sparse import issparse

prep = pipe_final.named_steps["prep"]
X_train_prep = prep.transform(X_train)          # reutiliza el preprocesador ya ajustado; no es un nuevo fit
nombres_prep = prep.get_feature_names_out()

print(f"Columnas antes del preprocesamiento: {X_train.shape[1]} "
      f"({len(COLS_NUMERICAS)} numéricas + {len(COLS_CATEGORICAS)} categóricas)")
print(f"Columnas después: {X_train_prep.shape[1]} "
      f"(numéricas escaladas + categóricas codificadas con one-hot)")

idx_num = [i for i, n in enumerate(nombres_prep) if n.startswith("num__")]
bloque_num = X_train_prep[:, idx_num]
bloque_num = bloque_num.toarray() if issparse(bloque_num) else bloque_num
print("\nMedia tras StandardScaler, primeras 5 numéricas (debe ser ~0):", np.round(bloque_num.mean(axis=0)[:5], 3))
print("Desv. estándar tras StandardScaler, primeras 5 (debe ser ~1):   ", np.round(bloque_num.std(axis=0)[:5], 3))

print("\nPrimeras columnas tras la transformación (vista previa):")
preview = X_train_prep[:5, :10]
preview = preview.toarray() if issparse(preview) else preview
pd.DataFrame(preview, columns=nombres_prep[:10])


Columnas antes del preprocesamiento: 42 (39 numéricas + 3 categóricas)
Columnas después: 184 (numéricas escaladas + categóricas codificadas con one-hot)

Media tras StandardScaler, primeras 5 numéricas (debe ser ~0): [-0.  0.  0. -0. -0.]
Desv. estándar tras StandardScaler, primeras 5 (debe ser ~1):    [1. 1. 1. 1. 1.]

Primeras columnas tras la transformación (vista previa):


,num__dur,num__spkts,num__dpkts,num__sbytes,num__dbytes,num__rate,num__sttl,num__dttl,num__sload,num__dload
0,-0.251601,0.104698,0.056469,-0.041757,-0.014094,-0.298806,-1.123874,-0.913097,-0.223969,1.305500
1,-0.255429,0.117732,0.141290,-0.043064,0.022467,-0.261629,-1.123874,-0.913097,-0.219678,4.670563
2,-0.258697,-0.155976,-0.183860,-0.047342,-0.106251,1.893040,0.964755,-1.160412,9.754459,-0.331286
3,0.067178,-0.103841,-0.127312,-0.054306,-0.104353,-0.322280,0.964755,0.988670,-0.228292,-0.330623
4,-0.138057,0.521777,0.289729,0.588904,-0.086786,-0.319213,-1.142606,-0.896041,-0.217464,-0.310642


## 7. Evaluación en test

In [16]:
pred = pipe_final.predict(X_test)

metricas_test = {
    "precision": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred),
}
print({k: round(v, 4) for k, v in metricas_test.items()})
print()
print(classification_report(y_test, pred, target_names=["normal (0)", "ataque (1)"], digits=4))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=["normal", "ataque"]).plot(cmap="Blues", values_format=",")
plt.title(f"Matriz de confusión en test — {mejor}")
plt.show()


{'precision': 0.9574, 'recall': 0.9324, 'f1': 0.9447}

              precision    recall  f1-score   support

  normal (0)     0.9616    0.9760    0.9687      6842
  ataque (1)     0.9574    0.9324    0.9447      3949

    accuracy                         0.9601     10791
   macro avg     0.9595    0.9542    0.9567     10791
weighted avg     0.9600    0.9601    0.9599     10791



/tmp/ipykernel_106972/851674541.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Serialización del Pipeline completo y metadatos

Se guarda el `Pipeline` entero (preprocesamiento + modelo) en un único archivo, para que en producción baste con `joblib.load(...).predict(df_nuevo)`. El JSON de metadatos documenta el contrato de entrada.

In [17]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
fecha = datetime.date.today().isoformat()
ruta_modelo = MODELS_DIR / f"modelo_{VERSION_MODELO}_{fecha}.joblib"
ruta_meta = MODELS_DIR / f"metadatos_{VERSION_MODELO}_{fecha}.json"

joblib.dump(pipe_final, ruta_modelo, compress=3)
tam_mb = ruta_modelo.stat().st_size / 1e6

metadatos = {
    "version": VERSION_MODELO,
    "fecha": fecha,
    "archivo_modelo": ruta_modelo.name,
    "estimador": mejor,
    "parametros_estimador": {k: repr(v) for k, v in modelos[mejor].get_params().items()},
    "dataset": DATA_PATH.name,
    "target": TARGET,
    "clases": {"0": "normal", "1": "ataque"},
    "columnas_excluidas": COLUMNAS_EXCLUIR,
    "columnas_entrada": {
        **{c: {"tipo": "numerica", "dtype": str(X_train[c].dtype)} for c in COLS_NUMERICAS},
        **{c: {"tipo": "categorica", "dtype": str(X_train[c].dtype),
               "categorias_train": sorted(X_train[c].dropna().astype(str).unique().tolist())}
           for c in COLS_CATEGORICAS},
    },
    "orden_columnas": COLS_ENTRADA,
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "estratificado": True,
              "n_train": len(X_train), "n_test": len(X_test)},
    "metricas_cv_f1": {k: {"media": float(v.mean()), "std": float(v.std())} for k, v in resultados_cv.items()},
    "metricas_test": {**{k: float(v) for k, v in metricas_test.items()},
                      "classification_report": classification_report(y_test, pred, output_dict=True)},
    "versiones": {"python": sys.version.split()[0], "sklearn": sklearn.__version__,
                  "pandas": pd.__version__, "numpy": np.__version__, "joblib": joblib.__version__},
}
with open(ruta_meta, "w", encoding="utf-8") as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)

print(f"Modelo: {ruta_modelo} ({tam_mb:.1f} MB)")
print(f"Metadatos: {ruta_meta}")
if tam_mb > MAX_MB_MODELO:
    print(f"AVISO: el modelo pesa más de {MAX_MB_MODELO} MB; reducir n_estimators/max_depth antes de subirlo.")


Modelo: models/modelo_v1_2026-09-29.joblib (4.2 MB)
Metadatos: models/metadatos_v1_2026-09-29.json


## 9. Comprobación: recargar el modelo guardado y predecir

Se simula el uso en producción: se carga el archivo y se predice sobre 3 filas de test usando solo las columnas del contrato.

In [18]:
modelo_cargado = joblib.load(ruta_modelo)
with open(ruta_meta, encoding="utf-8") as f:
    meta_cargada = json.load(f)

muestra = X_test.head(3)[meta_cargada["orden_columnas"]]
pred_muestra = modelo_cargado.predict(muestra)
prob_muestra = modelo_cargado.predict_proba(muestra)[:, 1]

print(pd.DataFrame({"prediccion": pred_muestra, "prob_ataque": prob_muestra.round(4),
                    "real": y_test.head(3).values}, index=muestra.index))
assert (pred_muestra == pipe_final.predict(muestra)).all(), "El modelo recargado no coincide"
print("\nOK: el modelo recargado reproduce las predicciones.")


       prediccion  prob_ataque  real
28153           1       0.9517     1
14799           0       0.0000     0
11323           0       0.0001     0

OK: el modelo recargado reproduce las predicciones.


## 10. Conclusiones

- **Modelo seleccionado:** Random Forest (`n_estimators=100, max_depth=20, min_samples_leaf=2`), elegido por validación cruzada estratificada (3 folds) sobre train: F1 = 0,936 ± 0,004, frente a F1 = 0,848 ± 0,006 de la regresión logística. El Random Forest maneja mejor las relaciones no lineales entre las variables de tráfico y no requiere que estén en la misma escala.
- **Métricas en test** (10.791 flujos, nunca vistos durante el entrenamiento ni la selección del modelo): precision 0,957, recall 0,932, F1 0,945.
- **Errores más costosos:** los falsos negativos (ataques clasificados como tráfico normal), porque implican que una intrusión pasa desapercibida. Con un recall de 0,932, alrededor del 7 % de los ataques del conjunto de test no se detectan; ver la matriz de confusión de la sección 7 para el conteo exacto.
- **Limitaciones:**
  - El dataset es de 2015 y sintético (generado con IXIA PerfectStorm); patrones de ataque más recientes podrían no estar representados.
  - `proto` tiene muchas categorías con muy pocas observaciones, lo que puede hacer que el modelo generalice peor sobre protocolos raros.
  - El desbalance de clases (63,4 % / 36,6 %) es moderado y no se corrigió con técnicas de remuestreo; F1 en la clase minoritaria (ataque) sigue siendo alto, pero conviene vigilarlo si el desbalance fuera mayor en producción.
- **Siguientes pasos (fases 2-4):** empaquetar este Pipeline en `train.py`/`predict.py`, contenerizarlo, exponerlo mediante una API REST y registrar sus predicciones para monitorear posible *data drift* frente a tráfico real.